In [3]:
import numpy as np
import jax
import jaxlib
import jax.numpy as jnp
import flax
import flax.linen as nn
import optax
from typing import Tuple, Callable, Any, Dict, Optional
import numpy.typing as npt
import copy
import pathlib
import matplotlib.pyplot as plt
import time
import json
import ast
import netket as nk
import os
import glob

#os.chdir("/home/ihuarte/Escritorio/Ivan/NN/")

# from ATMOS_VA.VA_project.src.VA_project.model.model import OxalateJKGamma
# from ATMOS_VA.VA_project.src.VA_project.engine.runners import Runner

#from NN_utils import load_vstate
#from correlations import correlations_vstate

jax.config.update("jax_enable_x64", True)
jax.config.update("jax_platform_name", "cpu")
jax.devices()

[CpuDevice(id=0)]

In [ ]:
def polyphase_components(x, strides):

    B, H, W, C = x.shape
    Hd, Wd = H // strides[0], W // strides[1], 
    # print(f"(B, H, W, C) = {x.shape}")
    # print(f"strides: {strides}")
    # print(f"Hd, Wd = {Hd}, {Wd}")
    x = x.reshape(
        (B, Wd, strides[1], Hd, strides[0], C),
        order='C').transpose((0, 1, 3, 2, 4, 5)
        ).reshape(B, Hd*Wd, *strides, C)
    return x

def get_maxnorm_indices(x, strides):
    """
    This function returns the traslation indices for a batched input of shape (B, H, W, C).
    It computes the polyphase components of a grid for each batch and channel, calculates 
    the L2 norm for each component and chooses the indices of the maximum value component.
    It works for both 1D and 2D inputs.
    Input:
        - x: (jnp.ArrayLike) Input data.
        - strides: (tuple) Strides for the next downsampling convolution.

    Returns:
        - x: Shifts (translations) for all batches and channels which makes the input 
             traslationaly equivariant.

    """
    _, H, W, _ = x.shape
    assert (H % strides[0]==0) & (W % strides[1]==0), f"`lattice_size` must be disible by `strides`. But they are {(H,W)} and {strides}"
    

    poly_comp = polyphase_components(x, strides).transpose((0,3,2,1,4))
    norm = jnp.linalg.norm(poly_comp, axis=-2, keepdims=True).squeeze(-2).transpose((0,3,1,2))
    norm = norm.reshape(*norm.shape[0:2],norm.shape[2] * norm.shape[3])
    maxnorm_idx = jnp.argmax(norm, axis=-1)[:,:,None]
    row_idx, col_idx = jnp.unravel_index(maxnorm_idx, strides)

    return -jnp.array([row_idx, col_idx]).squeeze(-1).transpose((1,2,0))


def polyphase_equivariance_adapter(x,strides):

    shifts = get_maxnorm_indices(x, strides)
    traslations = lambda x, shift: jnp.roll(x, shift=shift, axis=(0,1))
    batched_traslations = lambda x, shifts: jax.vmap(traslations)(x, shifts)
    
    x = jax.vmap(batched_traslations)(x.transpose((0,3,1,2)), shifts)

    return x.transpose((0,2,3,1))


x=jnp.array([1,2,3,4,5,6]).reshape()



In [1]:
import jax
import jax.numpy as jnp
from functools import partial
from NN_module.models.CvT_APS import Conv_APS


# supongamos Conv_APS es tu clase Flax y APS_equivariance_adapter ya integrada
model = Conv_APS(channels=16, kernel=(3,3), strides=(2,2), padding="CIRCULAR", use_bias=False)
rng = jax.random.PRNGKey(0)

# crea una entrada de prueba (B,H,W,C) o (B,C,H,W) según tu conv espera
x = jax.random.normal(rng, (4, 4, 4, 3))  # ejemplo B=4, C=3, H=W=32

# inicializa parámetros (depende de tu API exacta)
variables = model.init(rng, x)

# función de aplicación
apply_fn = lambda inp: model.apply(variables, inp)

# función de shift (desplazar por (dy,dx))
def shift(x, dy, dx):
    return jnp.roll(jnp.roll(x, shift=dy, axis=-2), shift=dx, axis=-1)

# prueba para varios desplazamientos t
for dy in range(0, 2):   # prueba con desplazamientos menores que stride
    for dx in range(0, 2):
        x_shift = shift(x, dy, dx)
        out1 = apply_fn(x_shift)
        out2 = shift(apply_fn(x), dy, dx)
        max_diff = jnp.max(jnp.abs(out1 - out2))
        print(f"shift ({dy},{dx}) max_diff = {max_diff}")
        assert jnp.allclose(out1, out2, atol=1e-6), "No es equivariante"

ValueError: Improper number of axes for norm: axis=(3, 4, 5). Pass one axis to compute a vector-norm, or two axes to compute a matrix-norm.

In [3]:
size = (4,4)
strides = (2,2)
assert (size[0] % strides[0]==0) & (size[1] % strides[1]==0), f"`lattice_size` must be disible by `strides`. But they are {size} and {strides}"
C = 8
B = 1

# key=jax.random.PRNGKey(977686)

# x_broad = jax.random.uniform(key, shape=(B,*size,C))
# fun = jax.jit(polyphase_equivariance_adapter(strides))
# fun(jax.random.uniform(key, shape=(B,*size,C)))

# %timeit -n 10000 -r 100 fun(jax.random.uniform(key, shape=(B,*size,C)))

In [5]:
import itertools
def get_channel_indices(H, W, C):
    print(H,W,C)
    assert (H*W)**C < 3e5, f"Too much channels or lattice size"
    
    grid_tras_ind = jnp.array([(i,j) for i in range(H) for j in range(W)])
    combo_idx = jnp.array(list(itertools.product(range(H*W), repeat=C)))
    return grid_tras_ind[combo_idx]

H = 2
W = 2
C = 5
get_channel_indices(H,W,C).shape[-2]

2 2 5


5

2 2 5


AssertionError: 

In [9]:
(H*W)**C > 3e5

False

In [6]:
3e5

300000.0

In [ ]:

_, ax = plt.subplots()
ax.hist(phase,rwidth=0.1)
ax.set_xlabel(r"$\varphi$")
ax.set_ylabel(r"$P(\varphi)$")

In [ ]:
set(phase),bins

In [ ]:
oxa=OxalateJKGamma(
    [4,5], 
    [0.2, 9.0, 72.0],
    **{'bc': 'periodic', 'order': 'default_2'}
)
E_ED, x_ED = Runner(oxa.cm).exact_energy_lanczos(k=5, eigenstates=True)

In [ ]:
art_path='/home/ihuarte/Escritorio/Ivan/NNs/Simulations/Oxalate_SplitTraining_ViT_CNN/Size_4x5/sweeps_0/Oxalate_SplitTraining_ViT_CNN_results_4x5_strength_0.2_theta_9.0_phi_72.0_channels_32_kernel_3x3_n_ffn_lay_1.json'

with open(art_path,'r') as f:
    artifact = json.load(f)

In [ ]:
x_ED = np.loadtxt("/home/ihuarte/Escritorio/Ivan/NNs/prueba/JKGamma/Split_oxalate_CNN/Oxalate_size_4x4/sweeps_20/Oxalate_xED_4x4_strength_0.2_theta_9.0_phi_72.0_ED.txt", dtype=complex)
x_vs = np.array(vstate.to_array())

In [ ]:



mod_vs, phase_vs, stats_vs = modphase(x_vs)
mod_ED, phase_ED, stats_ED = modphase(x_ED)



In [ ]:
a=(383,339)
print(f"|| {a} ||")

In [ ]:
import matplotlib.transforms as mtransforms
_,ax= plt.subplots(4,1, figsize=[15,10])

ax[0].set_title(r"$Modulus\;and\;Phase\qquad Oxalate\;size\;%d x %d \qquad a=%.1f\;\;\theta=%.1f \;\; \phi=%.1f$"%(4,4,0.2,9.0,72.0))
ax[0].set_xticks([])
ax[0].set_ylabel(r"$Modulus$")
ax[0].set_ylim(-0.01,max(max(mod_ED),max(mod_vs))*9/8)
ax[0].plot(mod_vs, alpha=0.6, color='r', label='vstate')
ax[0].plot(mod_ED, alpha=0.6, label='ED')
ax[0].legend()

ax[1].set_xticks([])
ax[1].set_yticks([-np.pi,-np.pi/2,0,np.pi/2,np.pi])
ax[1].set_yticklabels([r"$-\pi$",r"$-\pi/2$",r"$0$",r"$\pi/2$",r"$\pi$"])
ax[1].set_ylabel(r"$Phase \;vstate$")
ax[1].set_ylim(-np.pi-0.1, np.pi+0.1)
ax[1].plot(phase_vs, alpha=0.25, ls='', marker='o', ms=0.9, color='r', label='vstate')
ax[1].legend(loc='upper right')

ax[2].set_xlabel(r"$C_i$")
ax[2].set_ylabel(r"$Phase \;ED$")
ax[2].set_ylim(-np.pi-0.1, np.pi+0.1)
ax[2].set_yticks([-np.pi,-np.pi/2,0,np.pi/2,np.pi])
ax[2].set_yticklabels([r"$-\pi$",r"$-\pi/2$",r"$0$",r"$\pi/2$",r"$\pi$"])
ax[2].plot(phase_ED, alpha=0.25, ls='', marker='o', ms=0.9, label='ED')
ax[2].legend(loc='upper right')

ax[3].set_xlabel(r"$Phase\;(radians)$")
ax[3].set_ylabel(r"$Phase \;histogram$")
ax[3].hist(phase_ED, bins=1000, range=(-np.pi, np.pi), density=True, alpha=0.7, label='ED')
ax[3].hist(phase_vs, bins=1000, range=(-np.pi, np.pi), color='r', density=True, alpha=0.7, label='vstate')


transform = mtransforms.blended_transform_factory(ax[3].transData, ax[3].transAxes)
if stats_ED['peaks'] is not None:
    for peak in stats_ED['peaks']['values']:
        ax[3].text(peak-0.1, 0.9, r"%.2f"%peak, color='b', transform=transform, fontsize=8, alpha=0.7)

if stats_vs['peaks'] is not None:
    for peak in stats_vs['peaks']['values']:
        ax[3].text(peak-0.1, 0.9, r"%.2f"%peak, color='b', transform=transform, fontsize=8, alpha=0.7)
ax[3].legend()
plt.tight_layout()
plt.savefig(write_folder + files[j] + ".jpeg", dpi=600, bbox_inches="tight")

In [ ]:
_, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(lr_schedule, marker='o',ls=None, label='Learning Rate Schedule')
ax.set_xlabel('Epochs')
ax.set_ylabel('Learning Rate')

In [ ]:
from VA_project.lattice.lattice import Chain,Square
from VA_project.model.cm import GeneralNeighborCoupling

size=[4,5]

fields=[(0., 'X'), (0.5, 'Z'), (0.8, 'Y')]
couplings=[(0.,'ZZ','NN'),(2.,'YY','NN2')]

chain= Square(*size,bc='periodic', order= 'default_1')
ZZYY= GeneralNeighborCoupling(chain, fields, couplings)
chain.plot_lattice(1)

In [ ]:
from NN_module.NN_utils import traslations_2D,traslations_2D_scan,traslations_2D_vmap

In [ ]:
a=jnp.arange(20)
size=[4,5]
token_size=[2,1]
sub_lat=[size[0]//token_size[0], size[1]//token_size[1]]
# a.reshape((sub_lat[1],token_size[1],sub_lat[0],token_size[0]),
#                   order='C').transpose((0, 2, 1, 3)).reshape(-1,token_size[0]*token_size[1]).squeeze()
#traslations_2D_scan(a,size).reshape(-1,*size)
traslations_2D(a,size, memory=False)#.reshape(-1,*size)



In [ ]:
import sys
from pathlib import Path
sys.path.append(str("/home/ihuarte/Escritorio/Ivan/ATMOS_VA/VA_project/src"))

from VA_project.model.model import OxalateJKGamma
from VA_project.engine.runners import Runner
from NN_module.sim_utils import (
    load_vstate
)
files=[
    "/home/ihuarte/Escritorio/Ivan/NNs/Simulations/ViT_2D/Oxalate_size_4x4/targets/Oxalate_results_4x4_strength_0.2_theta_54.0_phi_0.0_b_2x2_Demb_128_heads_2_blocks_2_ffn_lay_2.json",
    "/home/ihuarte/Escritorio/Ivan/NNs/Simulations/ViT_2D/Oxalate_size_4x4/targets/Oxalate_results_4x4_strength_0.2_theta_54.0_phi_0.0_b_2x2_Demb_64_heads_8_blocks_2_ffn_lay_4.json",
    "/home/ihuarte/Escritorio/Ivan/NNs/Simulations/ViT_2D/Oxalate_size_4x4/targets/Oxalate_results_4x4_strength_0.2_theta_54.0_phi_0.0_b_2x2_Demb_64_heads_2_blocks_2_ffn_lay_4.json"
]

size=[4,4]
strength=0.2
theta=54.
phi=0.
kwargs_lattice={
    'bc':'periodic',
    'order':'default_2'
}

oxa=OxalateJKGamma(
            size, 
            [strength, theta, phi],
            **kwargs_lattice
        )
H = Runner(oxa.cm).build_hamiltonian()
E_ED, x_ED = Runner(oxa.cm).exact_energy_lanczos(eigenstates=True, k=150)
idx=np.argsort(E_ED)
E_ED=E_ED[idx]
x_ED=x_ED.T[idx,:]

In [ ]:
i=2
path_artifact= files[i]

with open(path_artifact,'r') as f:
    artifact = json.load(f)

vstate=load_vstate(artifact).to_array()
vstate=np.array(vstate, dtype=np.complex128)


In [ ]:
c=np.sum(x_ED.conjugate()*vstate, axis=1)
P=np.abs(c)**2
sum(P)

In [ ]:
E_best=artifact['results']['E_best']
e=-np.inf; idx=0
while e<E_best:
    e=E_ED[idx]
    idx+=1

idx, E_ED[idx],E_ED[idx-1]

In [ ]:
P_sum=sum(P)

_,ax=plt.subplots()
ax.set_title(f"Projections #{i}")
ax.set_xlabel("Eigenstates")
ax.set_ylabel("Probability")
ax.bar(range(len(P)), P, label=r"$\Sigma_i\; P_i$")
ax.vlines(idx, 0,max(P), color='r', ls='--', label=r"$\sim E_{sim}$")
ax.legend(fontsize=8)
ax.text(0.5, 0.8, r"$\Sigma_i\; P_i = %.4f$"%P_sum+"\n"+r"$E_1=%.3f$"%E_ED[0]+"\n"+r"$E_{150}=%.3f$"%(E_ED[-1])+"\n"+r"$E_{sim}=%.3f$"%E_best, transform=ax.transAxes, fontsize=10, bbox=dict(facecolor="white", alpha=0.4))
#ax.set_yscale('log')

In [ ]:
a=False
b=True
not a and b

# Learning Rate Scheduler

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import jax.numpy as jnp
from NN_module.NN_utils import scheduler_initializer

In [ ]:

schedule={
    'epochs': 500,
    'cos_exp_scheduler':{
        "lr0":0.6,
        "decay_exp":1.,
        "cosine_cycles":2.,
        "n":0.1,
        "lr_min": 0.01
    }
}

lr_sch=scheduler_initializer('cos_exp_scheduler', schedule)


# recta = lambda x: (lr_min-n)/epochs*x+n
# y_recta=recta(jnp.arange(0,epochs))

y=lr_sch(jnp.arange(0,schedule['epochs']))

_,ax= plt.subplots(1,1,figsize=(10,5))
ax.plot(y, label="exp decay")
#ax.plot(np.arange(epochs),y_recta, ls= '--', color='r')
ax.set_xlabel("step")
ax.set_ylabel("learning rate")
#ax.set_ylim(0, 1.05)
ax.legend()
ax.grid()

In [ ]:
21388438632/(1024)**3

In [ ]:
import jax.numpy as jnp
import jax
a= jnp.ones((4,4), dtype=complex)/10
b= jnp.ones((4,4) ,dtype=complex)


stack=jnp.array([a,b]).transpose(1,0,2)
z2_sym=jax.nn.logsumexp(
    stack,
    b=jnp.array([1., -1.])[None,:,None],
    axis=1
)
z2_2d_symm=jax.nn.logsumexp(
    z2_sym,
    axis=0
)

In [ ]:
from flax.serialization import to_bytes, from_bytes
import msgpack
file="/home/ihuarte/Escritorio/Ivan/NNs/Simulations/ViT_2D/Oxalate_size_4x4/Oxalate_vstate_params_4x4_strength_0.2_theta_90.0_phi_115.2_b_2x1_Demb_64_heads_4_blocks_2_ffn_lay_4.msgpack"

In [ ]:
with open(file, "rb") as f:
    bytes_data = f.read()

# Decodifica con msgpack sin un schema de Flax
unpacked = msgpack.unpackb(bytes_data, raw=False)

import pprint
pprint.pprint(unpacked)


In [ ]:
_, ax= plt.subplots()
x=np.arange(50,300)
ax.set_ylim(0,0.1)
ax.set_xlim(0,300)
ax.plot(x,0.05*0.987**(x-50))

In [ ]:
import numpy as np
file="/home/ihuarte/Escritorio/Ivan/NNs/Simulations/pruebas_Ising_LRM/LRM/ViT/Oxalate_xED_4x5_strength_0.2_theta_0_phi_0.txt"
x_ED= np.loadtxt(file, dtype=complex)
x_ED

In [ ]:
jnp.angle(x_ED)

In [ ]:
from NN_module.sim_utils import load_vstate
import json
file="/home/ihuarte/Escritorio/Ivan/NNs/Simulations/pruebas_Ising_LRM/LRM/ViT/Oxalate_results_4x5_strength_0.2_theta_0_phi_0_XZ_-1.0_0.001_J_1.5_alpha_2.5_b_2x1_Demb_32_heads_2_blocks_2_ffn_lay_2.json"

In [ ]:
with open(file,'rb') as f:
    conf=json.load(f)

vstate=load_vstate(conf)